# Snapshot Inicial: Ecosistema Politico en X (Twitter)

Recoleccion y analisis de tweets desde la inscripcion de candidaturas (13 de marzo 2026).

**Pasos:**
1. Resolver cuentas semilla (handles -> IDs)
2. Recolectar timelines de cuentas semilla
3. Buscar menciones de candidatos principales
4. Analizar hashtags y handles descubiertos

In [40]:
import sys
sys.path.insert(0, "..")

import polars as pl
import yaml
import json
import plotly.express as px

from src.config import DATA_RAW, DATA_PROCESSED, PROJECT_ROOT
from src.twitter_client import get_client, TWEET_FIELDS, EXPANSIONS, USER_FIELDS
from src.collectors.tweet_collector import (
    SINCE_DATE, collect_user_timeline, resolve_user_ids,
    save_tweets, search_tweets,
)

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

print(f"Recolectando desde: {SINCE_DATE}")

Recolectando desde: 2026-03-13T00:00:00Z


## 1. Cargar cuentas semilla y resolver IDs

Las cuentas estan en `config/candidates.yaml`, agrupadas por bloque (derecha/izquierda).

In [41]:
# Cargar configuracion y extraer todas las cuentas con IDs existentes
config_path = PROJECT_ROOT / "config" / "candidates.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Aplanar todas las cuentas (candidatos + asociadas) de todos los bloques
all_accounts = [
    account
    for bloc in config["bloques"].values()
    for key in ("candidatos", "cuentas_asociadas")
    for account in bloc.get(key, [])
]

# id_map: handle_sin_@ -> user_id (solo los ya guardados)
id_map = {
    a["handle"].lstrip("@"): a["id"]
    for a in all_accounts
    if a.get("id", "").strip()
}

print(f"Cuentas totales: {len(all_accounts)}, IDs en YAML: {len(id_map)}")
print(id_map)

Cuentas totales: 10, IDs en YAML: 10
{'PalomaValenciaL': '149281495', 'ABDELAESPRIELLA': '1657059045956517897', 'JDOviedoAr': '219434063', 'CeDemocratico': '1115440213', 'AlvaroUribeVel': '61097151', 'IvanCepedaCast': '98781946', 'petrogustavo': '49849732', 'PizarroMariaJo': '1362851972', 'GustavoBolivar': '50981729', 'PactoHistorico': '998190549353017345'}


In [42]:
# Resolver handles sin ID y guardar en YAML
handles_missing = [a["handle"] for a in all_accounts if a["handle"].lstrip("@") not in id_map]
print(f"Sin ID: {handles_missing}")

if handles_missing:
    new_ids = resolve_user_ids(handles_missing)  # retorna {username_sin_@: id}
    id_map.update(new_ids)

    for a in all_accounts:
        username = a["handle"].lstrip("@")
        if username in new_ids:
            a["id"] = new_ids[username]

    with open(config_path, "w") as f:
        yaml.dump(config, f, allow_unicode=True, default_flow_style=False, sort_keys=False)
    print(f"YAML actualizado: {new_ids}")

print(f"\nid_map completo ({len(id_map)}):")
for u, uid in sorted(id_map.items()):
    print(f"  @{u} -> {uid}")

Sin ID: []

id_map completo (10):
  @ABDELAESPRIELLA -> 1657059045956517897
  @AlvaroUribeVel -> 61097151
  @CeDemocratico -> 1115440213
  @GustavoBolivar -> 50981729
  @IvanCepedaCast -> 98781946
  @JDOviedoAr -> 219434063
  @PactoHistorico -> 998190549353017345
  @PalomaValenciaL -> 149281495
  @PizarroMariaJo -> 1362851972
  @petrogustavo -> 49849732


### 2a. Guardar bloque derecha desde memoria (sin re-consultar API)

`bloc_records` tiene los tweets de derecha que fallaron al guardar por tipos mixtos en IDs.
Normalizamos los IDs a string y guardamos directo.

In [43]:
# Bloque derecha: cargar desde parquet si ya existe, si no convertir desde JSON
_parquet_derecha = DATA_RAW / "seed_timelines_derecha.parquet"

if _parquet_derecha.exists():
    df_derecha = pl.read_parquet(_parquet_derecha)
    bloc_records = df_derecha.to_dicts()
    print(f"Cargado desde disco: {len(df_derecha)} tweets ({_parquet_derecha.name})")
else:
    import json as _json
    from datetime import datetime
    with open(DATA_RAW / "tweets_bloc2.json", encoding="utf-8") as f:
        bloc_records = _json.load(f)
    df_derecha = pl.DataFrame(bloc_records, infer_schema_length=None)
    df_derecha = df_derecha.with_columns(pl.lit(datetime.now().isoformat()).alias("collected_at"))
    df_derecha = df_derecha.unique(subset=["tweet_id"])
    _parquet_derecha.parent.mkdir(parents=True, exist_ok=True)
    df_derecha.write_parquet(_parquet_derecha)
    print(f"Convertido y guardado: {len(df_derecha)} tweets")

seed_tweets = {"derecha": bloc_records}

Cargado desde disco: 1412 tweets (seed_timelines_derecha.parquet)


### 2b. Recolectar solo bloque izquierda

Derecha ya esta guardado. Solo consultamos las cuentas de izquierda.

In [44]:
# Bloque izquierda: cargar desde parquet si ya existe, si no recolectar del API
_parquet_izq = DATA_RAW / "seed_timelines_izquierda.parquet"

if _parquet_izq.exists():
    df_izq = pl.read_parquet(_parquet_izq)
    izq_records = df_izq.to_dicts()
    print(f"Cargado desde disco: {len(df_izq)} tweets ({_parquet_izq.name})")
    seed_tweets["izquierda"] = izq_records
else:
    import json as _json
    from datetime import datetime
    izq_records = []
    for handle in blocs["izquierda"]:
        username = handle.lstrip("@")
        user_id = id_map.get(username)
        if not user_id:
            print(f"  No se pudo resolver {handle}, saltando")
            continue
        print(f"Recolectando @{username}... ", end="")
        records = collect_user_timeline(user_id)
        print(f"{len(records)} tweets")
        izq_records.extend(records)

    # Respaldo JSON
    with open(DATA_RAW / "tweets_izquierda.json", "w", encoding="utf-8") as f:
        _json.dump(izq_records, f, ensure_ascii=False, indent=4)

    df_izq = pl.DataFrame(izq_records, infer_schema_length=None)
    df_izq = df_izq.with_columns(pl.lit(datetime.now().isoformat()).alias("collected_at"))
    df_izq = df_izq.unique(subset=["tweet_id"])
    df_izq.write_parquet(_parquet_izq)
    print(f"Guardado: {len(df_izq)} tweets en {_parquet_izq.name}")
    seed_tweets["izquierda"] = izq_records

total = sum(len(v) for v in seed_tweets.values())
print(f"Total semillas: {total} tweets en {len(seed_tweets)} bloques")

Cargado desde disco: 961 tweets (seed_timelines_izquierda.parquet)
Total semillas: 2373 tweets en 2 bloques


## 3. Red de interaccion desde tweets existentes

En vez de consultar la lista de seguidos de cada semilla (costo: $0.01 por cuenta seguida),
construimos la red de interaccion a partir de los tweets ya recolectados.

Extraemos tres tipos de conexion:
- **Mentions**: a quien mencionan las semillas en sus tweets
- **Retweets/Quotes**: de quien amplifican contenido
- **Replies**: con quien conversan

Esto produce una red de interaccion real (mas util para THANOS que los follows estaticos)
y tiene **costo cero** porque usa datos ya descargados.

In [45]:
# Cargar todos los tweets recolectados
parquet_files = list(DATA_RAW.glob("*.parquet"))
df_all = pl.concat([pl.read_parquet(f) for f in parquet_files])
df_all = df_all.unique(subset=["tweet_id"])
print(f"Tweets totales: {len(df_all)}")

seed_handles = {h.lower() for h in id_map.keys()}

Tweets totales: 2373


### 3.1 Edges por mentions

In [46]:
edges_mentions = (
    df_all
    .select("author_username", "mentions")
    .explode("mentions")
    .drop_nulls("mentions")
    .rename({"author_username": "source", "mentions": "target"})
    .with_columns(
        pl.col("source").str.to_lowercase(),
        pl.col("target").str.to_lowercase(),
        pl.lit("mention").alias("edge_type"),
    )
)

### 3.2 Edges por retweets/quotes (ref_type -> author del tweet original)
Solo podemos vincular al autor original si esta en nuestros datos

In [47]:
ref_tweets = (
    df_all
    .filter(pl.col("ref_type").is_in(["retweeted", "quoted"]))
    .select("author_username", "ref_tweet_id", "ref_type")
)

Mapear ref_tweet_id -> autor original

In [48]:
tweet_author_map = df_all.select(
    pl.col("tweet_id").alias("ref_tweet_id"),
    pl.col("author_username").alias("original_author"),
)
edges_retweets = (
    ref_tweets
    .join(tweet_author_map, on="ref_tweet_id", how="inner")
    .select(
        pl.col("author_username").str.to_lowercase().alias("source"),
        pl.col("original_author").str.to_lowercase().alias("target"),
        pl.col("ref_type").alias("edge_type"),
    )
)

### 3.3 Edges por replies
in_reply_to_user_id -> necesitamos mapear a username

In [49]:
user_id_to_handle = df_all.select(
    pl.col("author_id").alias("in_reply_to_user_id"),
    pl.col("author_username").str.to_lowercase().alias("target"),
).unique(subset=["in_reply_to_user_id"])

edges_replies = (
    df_all
    .filter(pl.col("in_reply_to_user_id").is_not_null())
    .select("author_username", "in_reply_to_user_id")
    .join(user_id_to_handle, on="in_reply_to_user_id", how="inner")
    .select(
        pl.col("author_username").str.to_lowercase().alias("source"),
        pl.col("target"),
        pl.lit("reply").alias("edge_type"),
    )
)

In [50]:
# --- Combinar todas las aristas ---
edges = pl.concat([edges_mentions, edges_retweets, edges_replies])
# Eliminar self-loops
edges = edges.filter(pl.col("source") != pl.col("target"))

print(f"Aristas totales: {len(edges)}")
print(f"  Mentions: {len(edges_mentions)}")
print(f"  Retweets/Quotes: {len(edges_retweets)}")
print(f"  Replies: {len(edges_replies)}")
print(f"  Nodos unicos: {edges.select(pl.concat_list(['source', 'target']).alias('n')).explode('n').n_unique()}")

Aristas totales: 1849
  Mentions: 1971
  Retweets/Quotes: 383
  Replies: 24
  Nodos unicos: 529


### 3.4 Nodos mas conectados
in-degree: cuantas veces son mencionados/RT/replied

In [51]:
in_degree = (
    edges
    .group_by("target")
    .agg(
        pl.len().alias("in_degree"),
        pl.col("edge_type").filter(pl.col("edge_type") == "mention").len().alias("mentions_in"),
        pl.col("edge_type").filter(pl.col("edge_type").is_in(["retweeted", "quoted"])).len().alias("rt_in"),
        pl.col("edge_type").filter(pl.col("edge_type") == "reply").len().alias("replies_in"),
        pl.col("source").n_unique().alias("unique_sources"),
    )
    .sort("in_degree", descending=True)
)

# Marcar si es cuenta semilla
in_degree = in_degree.with_columns(
    pl.col("target").is_in(list(seed_handles)).alias("is_seed")
)

print("Top 30 cuentas mas referenciadas en la red:")
print(in_degree.head(30))

Top 30 cuentas mas referenciadas en la red:
shape: (30, 7)
┌─────────────────┬───────────┬─────────────┬───────┬────────────┬────────────────┬─────────┐
│ target          ┆ in_degree ┆ mentions_in ┆ rt_in ┆ replies_in ┆ unique_sources ┆ is_seed │
│ ---             ┆ ---       ┆ ---         ┆ ---   ┆ ---        ┆ ---            ┆ ---     │
│ str             ┆ u32       ┆ u32         ┆ u32   ┆ u32        ┆ u32            ┆ bool    │
╞═════════════════╪═══════════╪═════════════╪═══════╪════════════╪════════════════╪═════════╡
│ palomavalencial ┆ 193       ┆ 131         ┆ 62    ┆ 0          ┆ 7              ┆ true    │
│ alvarouribevel  ┆ 136       ┆ 75          ┆ 61    ┆ 0          ┆ 6              ┆ true    │
│ petrogustavo    ┆ 111       ┆ 71          ┆ 39    ┆ 1          ┆ 9              ┆ true    │
│ ivancepedacast  ┆ 97        ┆ 70          ┆ 27    ┆ 0          ┆ 7              ┆ true    │
│ aida_quilcue    ┆ 53        ┆ 53          ┆ 0     ┆ 0          ┆ 6              ┆ false   │
│

### 3.5 Cuentas NO semilla mas relevantes (candidatas a expandir red)

In [52]:
non_seed_top = in_degree.filter(~pl.col("is_seed")).head(30)
print("\nTop 30 cuentas NO semilla mas referenciadas (red organica):")
print(non_seed_top)



Top 30 cuentas NO semilla mas referenciadas (red organica):
shape: (30, 7)
┌─────────────────┬───────────┬─────────────┬───────┬────────────┬────────────────┬─────────┐
│ target          ┆ in_degree ┆ mentions_in ┆ rt_in ┆ replies_in ┆ unique_sources ┆ is_seed │
│ ---             ┆ ---       ┆ ---         ┆ ---   ┆ ---        ┆ ---            ┆ ---     │
│ str             ┆ u32       ┆ u32         ┆ u32   ┆ u32        ┆ u32            ┆ bool    │
╞═════════════════╪═══════════╪═════════════╪═══════╪════════════╪════════════════╪═════════╡
│ aida_quilcue    ┆ 53        ┆ 53          ┆ 0     ┆ 0          ┆ 6              ┆ false   │
│ revistasemana   ┆ 30        ┆ 30          ┆ 0     ┆ 0          ┆ 7              ┆ false   │
│ jrestrp         ┆ 27        ┆ 27          ┆ 0     ┆ 0          ┆ 1              ┆ false   │
│ diegoasantos    ┆ 19        ┆ 19          ┆ 0     ┆ 0          ┆ 2              ┆ false   │
│ juanmanuelgalan ┆ 16        ┆ 16          ┆ 0     ┆ 0          ┆ 4          

### 3.6 Visualizacion: top 20 por in-degree

In [53]:
plot_data = (
    in_degree.head(20)
    .to_pandas()
    .melt(
        id_vars=["target"],
        value_vars=["mentions_in", "rt_in", "replies_in"],
        var_name="tipo",
        value_name="interacciones",
    )
)

fig = px.bar(
    plot_data,
    x="target",
    y="interacciones",
    color="tipo",
    title="Top 20 cuentas por interacciones recibidas (desglose por tipo)",
    labels={"target": "Cuenta", "interacciones": "Interacciones", "tipo": "Tipo"},
    barmode="stack",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [54]:
# --- 3.7 Out-degree: quien interactua mas (detectar bots o cuentas de campana) ---
out_degree = (
    edges
    .group_by("source")
    .agg(
        pl.len().alias("out_degree"),
        pl.col("target").n_unique().alias("unique_targets"),
    )
    .sort("out_degree", descending=True)
)
print("\nTop 20 cuentas que mas interactuan (posibles bots/campana):")
print(out_degree.head(20))


Top 20 cuentas que mas interactuan (posibles bots/campana):
shape: (10, 3)
┌─────────────────┬────────────┬────────────────┐
│ source          ┆ out_degree ┆ unique_targets │
│ ---             ┆ ---        ┆ ---            │
│ str             ┆ u32        ┆ u32            │
╞═════════════════╪════════════╪════════════════╡
│ jdoviedoar      ┆ 407        ┆ 165            │
│ cedemocratico   ┆ 291        ┆ 43             │
│ ivancepedacast  ┆ 196        ┆ 85             │
│ pizarromariajo  ┆ 181        ┆ 72             │
│ abdelaespriella ┆ 155        ┆ 89             │
│ petrogustavo    ┆ 137        ┆ 87             │
│ alvarouribevel  ┆ 132        ┆ 58             │
│ palomavalencial ┆ 132        ┆ 57             │
│ pactohistorico  ┆ 111        ┆ 27             │
│ gustavobolivar  ┆ 107        ┆ 51             │
└─────────────────┴────────────┴────────────────┘


In [ ]:
# --- 3.8 Guardar red ---
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
edges.write_parquet(DATA_PROCESSED / "interaction_edges.parquet")
in_degree.write_parquet(DATA_PROCESSED / "interaction_in_degree.parquet")
print("\nGuardado: interaction_edges.parquet, interaction_in_degree.parquet")

## 4. Analisis de los datos recolectados

Cargamos todos los parquets recolectados y analizamos hashtags, handles y autores.

In [ ]:
# Cargar todos los parquets recolectados
parquet_files = list(DATA_RAW.glob("*.parquet"))
print(f"Archivos encontrados: {len(parquet_files)}")
for f in parquet_files:
    df_tmp = pl.read_parquet(f)
    print(f"  {f.name}: {len(df_tmp)} tweets")

df = pl.concat([pl.read_parquet(f) for f in parquet_files])
df = df.unique(subset=["tweet_id"])
print(f"\nTotal tweets unicos: {len(df)}")

### 4.1 Top hashtags

Estos son los hashtags mas usados en el ecosistema politico recolectado.
Los top 10 por bloque son los que alimentan la variable `x_{t,h,l}^(k)` del modelo THANOS.

In [ ]:
hashtags_df = (
    df.select("tweet_id", "hashtags")
    .explode("hashtags")
    .drop_nulls("hashtags")
    .group_by(pl.col("hashtags").str.to_lowercase().alias("hashtag"))
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(f"Hashtags unicos encontrados: {len(hashtags_df)}\n")
print("Top 30 hashtags:")
hashtags_df.head(30)

### 4.2 Top handles mencionados

Cuentas mas mencionadas en los tweets recolectados. Candidatos a ser agregados al tracking.

In [ ]:
mentions_df = (
    df.select("tweet_id", "mentions")
    .explode("mentions")
    .drop_nulls("mentions")
    .group_by(pl.col("mentions").str.to_lowercase().alias("handle"))
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print(f"Handles unicos mencionados: {len(mentions_df)}\n")
print("Top 30 handles mencionados:")
mentions_df.head(30)

### 4.3 Autores mas activos

Usuarios que mas publican sobre los candidatos. Utiles para identificar influenciadores
y tambien posibles bots (cuentas con actividad anormalmente alta).

In [ ]:
authors_df = (
    df.group_by("author_username")
    .agg(
        pl.len().alias("tweet_count"),
        pl.col("retweet_count").sum().alias("total_retweets"),
        pl.col("like_count").sum().alias("total_likes"),
        pl.col("author_followers").first().alias("followers"),
        pl.col("author_created_at").first().alias("account_created"),
    )
    .sort("tweet_count", descending=True)
)

print("Top 30 autores mas activos:")
authors_df.head(30)

### 4.4 Volumen de tweets por dia

Visualizacion del volumen de actividad diaria desde la inscripcion de candidaturas.

In [ ]:
import plotly.express as px

daily = (
    df.with_columns(pl.col("created_at").str.slice(0, 10).alias("date"))
    .group_by("date")
    .agg(pl.len().alias("tweets"))
    .sort("date")
)

fig = px.bar(
    daily.to_pandas(),
    x="date", y="tweets",
    title="Volumen diario de tweets recolectados",
    labels={"date": "Fecha", "tweets": "Tweets"},
)
fig.show()

### 4.5 Distribucion de tipos de tweet

Proporcion de tweets originales, retweets, replies y quotes.

In [ ]:
type_counts = (
    df.with_columns(
        pl.col("ref_type").fill_null("original").alias("tweet_type")
    )
    .group_by("tweet_type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

fig = px.pie(
    type_counts.to_pandas(),
    values="count", names="tweet_type",
    title="Distribucion por tipo de tweet",
)
fig.show()

## 4b. Expansion de red: desde las interacciones detectadas

Usamos las cuentas NO semilla mas referenciadas (seccion 3) como candidatas a expansion.
Buscamos tweets de esas cuentas usando hashtags de campana (busqueda, no timeline).

**Estimacion de costo antes de ejecutar.**

In [ ]:
from src.cost_estimator import estimate_search, estimate_user_lookup, print_estimate

# Cuentas no-semilla mas relevantes de la red de interaccion
expansion_candidates = non_seed_top.head(20).get_column("target").to_list()
print(f"Candidatas a expansion ({len(expansion_candidates)}): {expansion_candidates[:10]}...")

# Top hashtags (minimo 5 apariciones)
TOP_N_HASHTAGS = 20
MIN_HASHTAG_COUNT = 5
top_hashtags = (
    hashtags_df
    .filter(pl.col("count") >= MIN_HASHTAG_COUNT)
    .head(TOP_N_HASHTAGS)
    .get_column("hashtag")
    .to_list()
)
print(f"Hashtags de campana ({len(top_hashtags)}): {top_hashtags}")

# --- Estimar costo ANTES de ejecutar ---
# Una busqueda por grupo de hashtags, max 5 paginas = ~500 tweets
est_search = estimate_search(query_count=1, tweets_per_query=500)
# Lookup de las cuentas candidatas para obtener metricas publicas
est_users = estimate_user_lookup(len(expansion_candidates))

BUDGET = 10.0  # USD - ajustar segun presupuesto disponible
total = print_estimate(est_search, est_users, budget=BUDGET)

if total > BUDGET:
    print("\n** Costo excede presupuesto. Ajustar parametros antes de continuar. **")

In [ ]:
from importlib import reload
import src.collectors.tweet_collector as tc
reload(tc)

# Buscar tweets con hashtags de campana
hashtag_query = " OR ".join(f"#{h}" for h in top_hashtags)
query = f"({hashtag_query[:400]}) lang:es -is:retweet"
print(f"Query ({len(query)} chars): {query[:200]}...")

search_tweets = tc.search_tweets(query, max_pages=5, use_full_archive=True)

# Filtrar por cuentas de la red (semillas + candidatas de expansion)
allowed = seed_handles | {c.lower() for c in expansion_candidates}
filtered = [t for t in search_tweets if (t.get("author_username") or "").lower() in allowed]

print(f"\nTweets encontrados: {len(search_tweets)}")
print(f"De cuentas en la red: {len(filtered)}")

tc.save_tweets(filtered, "expanded_network_tweets")

## 5. Guardar rankings

Almacena los rankings calculados en `data/processed/` para uso posterior en el modelo.

In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

hashtags_df.write_parquet(DATA_PROCESSED / "hashtag_rankings.parquet")
mentions_df.write_parquet(DATA_PROCESSED / "mention_rankings.parquet")
authors_df.write_parquet(DATA_PROCESSED / "author_rankings.parquet")

print("Rankings guardados en data/processed/:")
print(f"  hashtag_rankings.parquet  ({len(hashtags_df)} hashtags)")
print(f"  mention_rankings.parquet  ({len(mentions_df)} handles)")
print(f"  author_rankings.parquet   ({len(authors_df)} autores)")

## 6. Exploracion libre

Celda para consultas ad-hoc sobre los datos recolectados.

In [ ]:
# Ejemplo: tweets con mas engagement
df.select(
    "author_username", "text", "retweet_count", "like_count", "created_at"
).sort("like_count", descending=True).head(20)

In [ ]:
# Ejemplo: buscar un hashtag especifico
# hashtag_a_buscar = "colombia"
# df.filter(
#     pl.col("hashtags").list.contains(hashtag_a_buscar)
# ).select("author_username", "text", "created_at").head(10)

In [55]:
import networkx as nx
from pyvis.network import Network

# Limitar a interacciones entre los top 100 nodos más referenciados
top_nodes = set(in_degree.head(100)["target"].to_list())

edges_sub = (
    edges
    .filter(
        pl.col("target").is_in(top_nodes) | pl.col("source").is_in(top_nodes)
    )
    .select(["source", "target", "edge_type"])
    .to_pandas()
)

G = nx.from_pandas_edgelist(
    edges_sub,
    source="source",
    target="target",
    edge_attr="edge_type",
    create_using=nx.DiGraph(),
)

# Tamaño de nodo proporcional al in-degree
in_deg = dict(G.in_degree())
max_deg = max(in_deg.values()) if in_deg else 1

net = Network(height="700px", width="100%", directed=True, notebook=True, cdn_resources="inline")
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=100)

for node in G.nodes():
    size = 5 + 40 * in_deg.get(node, 0) / max_deg
    is_seed = node in seed_handles
    color = "#e74c3c" if is_seed else "#3498db"
    net.add_node(node, label=node, size=size, color=color, title=f"{node}\nin_degree: {in_deg.get(node,0)}")

color_map = {"mention": "#95a5a6", "retweeted": "#e67e22", "quoted": "#9b59b6", "reply": "#2ecc71"}
for src, tgt, data in G.edges(data=True):
    etype = data.get("edge_type", "mention")
    net.add_edge(src, tgt, color=color_map.get(etype, "#95a5a6"), width=0.5)

net.show("red_interacciones.html")


AssertionError: cdn_resources not in [local, in_line, remote].

In [57]:
import networkx as nx
import yaml

with open("../config/candidates.yaml") as f:
    config = yaml.safe_load(f)

def extract_handles(bloque):
    handles = set()
    for c in bloque.get("candidatos", []):
        handles.add(c["handle"].lstrip("@").lower())
    for c in bloque.get("cuentas_asociadas", []):
        handles.add(c["handle"].lstrip("@").lower())
    return handles

derecha = extract_handles(config["bloques"]["derecha"])
izquierda = extract_handles(config["bloques"]["izquierda"])

G = nx.from_pandas_edgelist(
    edges.to_pandas(),
    source="source",
    target="target",
    edge_attr="edge_type",
    create_using=nx.DiGraph(),
)

in_deg = dict(in_degree.select(["target", "in_degree"]).iter_rows())
nx.set_node_attributes(G, in_deg, "in_degree")

def tendencia(node):
    n = node.lower()
    if n in derecha:
        return "derecha"
    if n in izquierda:
        return "izquierda"
    return "desconocido"

nx.set_node_attributes(G, {n: tendencia(n) for n in G.nodes()}, "tendencia")

output_path = DATA_PROCESSED / "red_interacciones.gexf"
nx.write_gexf(G, output_path)
print(f"Exportado: {output_path} ({G.number_of_nodes()} nodos, {G.number_of_edges()} edges)")
print(f"Derecha: {derecha}")
print(f"Izquierda: {izquierda}")


Exportado: /mnt/storage/proyectos/entendiendo_dinamica_politica/notebooks/../data/processed/red_interacciones.gexf (529 nodos, 734 edges)
Derecha: {'palomavalencial', 'cedemocratico', 'alvarouribevel', 'abdelaespriella', 'jdoviedoar'}
Izquierda: {'pactohistorico', 'ivancepedacast', 'petrogustavo', 'pizarromariajo', 'gustavobolivar'}
